In [15]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, round

spark = SparkSession.builder \
    .appName("Banking Transaction Analysis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Ready! Version:", spark.version)

Spark Ready! Version: 4.1.1


In [16]:
# Load the dataset
df = spark.read.csv(
    "banking_transactions.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully!")
print("Total Records:", df.count())
df.printSchema()
df.show(10)

Dataset Loaded Successfully!
Total Records: 50
root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location: string (nullable = true)
 |-- status: string (nullable = true)

+--------------+----------+-------------+----------------+---------+-------------------+----------+-------+
|transaction_id|account_id|customer_name|transaction_type|   amount|          timestamp|  location| status|
+--------------+----------+-------------+----------------+---------+-------------------+----------+-------+
|      TXN00001|   ACC0002|    Sara Khan|        TRANSFER|  4239.11|2024-05-05 07:08:00|   Karachi|PENDING|
|      TXN00002|   ACC0002|    Sara Khan|         DEPOSIT| 88778.63|2024-01-16 02:13:00|Rawalpindi|PENDING|
|      TXN00003|   ACC0010|   Sana Iqbal|      WITHDRAWAL|  4467.1

In [17]:
# Basic Statistics
print("="*50)
print("BASIC STATISTICS:")
print("="*50)
df.describe("amount").show()

BASIC STATISTICS:
+-------+------------------+
|summary|            amount|
+-------+------------------+
|  count|                50|
|   mean| 70939.01340000001|
| stddev|46041.734181867716|
|    min|           4239.11|
|    max|         144672.25|
+-------+------------------+



In [18]:
# Detect High Value Transactions
THRESHOLD = 40000

print("="*50)
print(f"HIGH-VALUE TRANSACTIONS (Amount > {THRESHOLD})")
print("="*50)

suspicious_df = df.filter(col("amount") > THRESHOLD)
suspicious_df.show(50)

print(f"Total Suspicious Transactions Found: {suspicious_df.count()}")

HIGH-VALUE TRANSACTIONS (Amount > 40000)
+--------------+----------+-------------+----------------+---------+-------------------+----------+-------+
|transaction_id|account_id|customer_name|transaction_type|   amount|          timestamp|  location| status|
+--------------+----------+-------------+----------------+---------+-------------------+----------+-------+
|      TXN00002|   ACC0002|    Sara Khan|         DEPOSIT| 88778.63|2024-01-16 02:13:00|Rawalpindi|PENDING|
|      TXN00004|   ACC0008|   Hina Javed|         DEPOSIT| 88595.22|2024-03-22 22:27:00|    Multan|SUCCESS|
|      TXN00006|   ACC0006|  Ayesha Noor|         DEPOSIT| 90757.04|2024-08-23 17:07:00|Faisalabad|SUCCESS|
|      TXN00007|   ACC0009|   Zaid Mirza|        TRANSFER| 44330.15|2024-10-22 06:45:00|   Karachi|SUCCESS|
|      TXN00008|   ACC0004| Fatima Malik|         DEPOSIT|116073.72|2024-04-29 03:24:00|  Sargodha| FAILED|
|      TXN00010|   ACC0003|   Ahmed Raza|      WITHDRAWAL| 80353.91|2024-03-24 14:24:00|  Sargo

In [11]:
#group by ACCOUNT_id

account_group = df.groupBy("account_id", "customer_name").agg(
    count("transaction_id").alias("total_transactions"),  
    spark_sum("amount").alias("total_amount"),        
    avg("amount").alias("avg_amount")                     
)

account_group.show()

+----------+-------------+------------------+------------------+------------------+
|account_id|customer_name|total_transactions|      total_amount|        avg_amount|
+----------+-------------+------------------+------------------+------------------+
|   ACC0005|  Usman Tariq|                 4|226836.22000000003| 56709.05500000001|
|   ACC0001|   Ali Hassan|                 5|490133.67000000004| 98026.73400000001|
|   ACC0003|   Ahmed Raza|                 6|341749.43000000005| 56958.23833333334|
|   ACC0008|   Hina Javed|                 5|         273913.63|         54782.726|
|   ACC0004| Fatima Malik|                 4|407822.04000000004|101955.51000000001|
|   ACC0010|   Sana Iqbal|                 3|         113638.84|37879.613333333335|
|   ACC0007|   Bilal Shah|                 4|         417390.44|         104347.61|
|   ACC0009|   Zaid Mirza|                 8|         605021.47|       75627.68375|
|   ACC0002|    Sara Khan|                 8|         545824.89|       68228

In [19]:
# Filter Completed Suspicious Transactions
print("="*50)
print("SUSPICIOUS + COMPLETED STATUS ONLY:")
print("="*50)

suspicious_completed = suspicious_df.filter(col("status") == "completed")
suspicious_completed.show()

print(f"Total Completed Suspicious Transactions: {suspicious_completed.count()}")

SUSPICIOUS + COMPLETED STATUS ONLY:
+--------------+----------+-------------+----------------+------+---------+--------+------+
|transaction_id|account_id|customer_name|transaction_type|amount|timestamp|location|status|
+--------------+----------+-------------+----------------+------+---------+--------+------+
+--------------+----------+-------------+----------------+------+---------+--------+------+

Total Completed Suspicious Transactions: 0


In [20]:
# Group by Account and Calculate Total Balance
print("="*50)
print("ACCOUNT-WISE SUMMARY:")
print("="*50)

account_summary = df.groupBy("account_id") \
    .agg(
        count("transaction_id").alias("total_transactions"),
        round(sum("amount"), 2).alias("total_balance")
    ) \
    .orderBy("total_balance", ascending=False)

account_summary.show(50)

ACCOUNT-WISE SUMMARY:
+----------+------------------+-------------+
|account_id|total_transactions|total_balance|
+----------+------------------+-------------+
|   ACC0009|                 8|    605021.47|
|   ACC0002|                 8|    545824.89|
|   ACC0001|                 5|    490133.67|
|   ACC0007|                 4|    417390.44|
|   ACC0004|                 4|    407822.04|
|   ACC0003|                 6|    341749.43|
|   ACC0008|                 5|    273913.63|
|   ACC0005|                 4|    226836.22|
|   ACC0006|                 3|    124620.04|
|   ACC0010|                 3|    113638.84|
+----------+------------------+-------------+



In [21]:
# Flag Suspicious Accounts (Total Balance > 500,000)
print("="*50)
print("SUSPICIOUS ACCOUNTS (Total Balance > 500,000):")
print("="*50)

suspicious_accounts = account_summary.filter(col("total_balance") > 500000)
suspicious_accounts.show()

print(f"Total Suspicious Accounts: {suspicious_accounts.count()}")

SUSPICIOUS ACCOUNTS (Total Balance > 500,000):
+----------+------------------+-------------+
|account_id|total_transactions|total_balance|
+----------+------------------+-------------+
|   ACC0009|                 8|    605021.47|
|   ACC0002|                 8|    545824.89|
+----------+------------------+-------------+

Total Suspicious Accounts: 2


In [22]:
# Save Results to CSV
print("Saving results...")

# Save suspicious transactions
suspicious_df.toPandas().to_csv("suspicious_transactions.csv", index=False)
print("suspicious_transactions.csv saved!")

# Save account summary
account_summary.toPandas().to_csv("account_summary.csv", index=False)
print("account_summary.csv saved!")

# Stop Spark Session
spark.stop()
print("="*50)
print("Analysis Complete! All files saved successfully!")
print("="*50)

Saving results...
suspicious_transactions.csv saved!
account_summary.csv saved!
Analysis Complete! All files saved successfully!
